In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sqlite3
from IPython.display import display, Markdown
from pathlib import Path
import os
import math

# AISYNPHYS stuff
import pyqtgraph
import neuroanalysis
import neuroanalysis.util
import aisynphys
import aisynphys.database

# Prints
print(aisynphys.__file__)
print(pyqtgraph.__file__)
print(neuroanalysis.__file__)

# Load matplotlib style
plt.style.use('plos.mplstyle')

# File path handling
# SCRIPT_DIR = Path(__file__).resolve().parent  # .py
SCRIPT_DIR = Path.cwd()  # .ipynb
FIGURES_DIR = SCRIPT_DIR / "figures_analysis"
print(f"Figure path: {FIGURES_DIR}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------

# Load DB
from aisynphys.database import SynphysDatabase # https://aisynphys.readthedocs.io/en/current-release/database_access.html#database-access



CACHE_PATH = Path("data/aisynphys/cache").resolve()  # Define cache directory
CACHE_PATH.mkdir(parents=True, exist_ok=True)  # Create a cache directory if it does not exist
aisynphys.config.cache_path = CACHE_PATH  # Tell AISynPhys to use this cache
DATA_ROOT = Path("data/aisynphys/cache")  # Make cache
SynphysDatabase.list_versions()  # List all available versions


# DB_VERSION = 'synphys_r2.1_medium.sqlite'  # Medium database
DB_VERSION = 'synphys_r2.1_full.sqlite'  # Full database

db = SynphysDatabase.load_version(DB_VERSION)  # Load DB
print(f"Loaded version: {DB_VERSION}")

# ------------------------------------------------------------
# Set up connection


# DB_PATH = Path('data/aisynphys/cache/database/synphys_r2.1_medium.sqlite').resolve()  # Medium database
DB_PATH = Path('data/aisynphys/cache/database/synphys_r2.1_full.sqlite').resolve()  # Full database
assert DB_PATH.exists(), f"Database file not found: {DB_PATH}"  # Ensure existence


con = sqlite3.connect(DB_PATH)  # Create SQLite connection to SQLite database
con.create_function("SQRT", 1, math.sqrt)   # now SQLite understands SQRT()
cur = con.cursor()  # database cursor
print(f"Connection set up to {DB_PATH}")

C:\Users\snrambo\aisynphys\aisynphys\__init__.py
c:\Users\snrambo\AppData\Local\miniconda3\envs\aisynphys39\lib\site-packages\pyqtgraph\__init__.py
C:\Users\snrambo\neuroanalysis\neuroanalysis\__init__.py
Figure path: c:\Users\snrambo\Projects\NEVR3901\HeterogenousRNN\HeterogeneousRNNs\data_analysis\figures_analysis
Loaded version: synphys_r2.1_full.sqlite
Connection set up to C:\Users\snrambo\Projects\NEVR3901\HeterogenousRNN\HeterogeneousRNNs\data_analysis\data\aisynphys\cache\database\synphys_r2.1_full.sqlite


Loading all matrices

In [ ]:
query = """ 
            SELECT 
                p.experiment_id,
                COUNT(*) AS total_pairs,
                SUM(p.has_synapse) AS num_synapses,
                COUNT(*) - SUM(p.has_synapse) AS num_no_synapses


            FROM pair p
            INNER JOIN cell       AS pre   ON p.pre_cell_id  = pre.id  
            INNER JOIN cell       AS post  ON p.post_cell_id = post.id
            INNER JOIN intrinsic  AS i_pre ON p.pre_cell_id  = i_pre.cell_id
            INNER JOIN intrinsic  AS i_post ON p.post_cell_id = i_post.cell_id
            LEFT JOIN synapse AS syn   ON syn.pair_id = p.id          -- can be NULL
            INNER JOIN experiment AS e     ON e.id = p.experiment_id
            INNER JOIN slice      AS s     ON e.slice_id = s.id


            WHERE 
                s.species = 'mouse'
                AND i_pre.fi_slope IS NOT NULL
                AND i_post.fi_slope IS NOT NULL
                AND p.has_synapse IS NOT NULL   -- only defined synapses
                AND syn.psp_amplitude IS NOT NULL
                
            GROUP BY p.experiment_id
            HAVING num_synapses > 1
            ;
        """

experiments = pd.read_sql_query(query, con)

unique_exp = experiments["experiment_id"].unique()
print(f"Number of experiments in the grouped DF: {len(unique_exp)}")

experiment_ids = experiments["experiment_id"].unique().tolist()

print(len(experiment_ids))

for exp in experiment_ids:
    # Test
    print(f"EXPERIMENT: {exp}")
    adj = np.load(f"psp_arrays/experiment_{exp}.npy")
    print(f"  PSP Matrix of shape: {adj.shape}")

# From here, can do motif analysis on PSP Matrix

Number of experiments in the grouped DF: 140
140
EXPERIMENT: 600
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 727
  PSP Matrix of shape: (4, 4)
EXPERIMENT: 793
  PSP Matrix of shape: (5, 5)
EXPERIMENT: 900
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 901
  PSP Matrix of shape: (5, 5)
EXPERIMENT: 1008
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 1036
  PSP Matrix of shape: (4, 4)
EXPERIMENT: 1077
  PSP Matrix of shape: (2, 2)
EXPERIMENT: 1124
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 1260
  PSP Matrix of shape: (2, 2)
EXPERIMENT: 1297
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 1436
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 1443
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 1461
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 1473
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 1477
  PSP Matrix of shape: (2, 2)
EXPERIMENT: 1485
  PSP Matrix of shape: (4, 4)
EXPERIMENT: 1488
  PSP Matrix of shape: (3, 3)
EXPERIMENT: 1511
  PSP Matrix of shape: (2, 2)
EXPERIMENT: 1521
  PSP Matrix of shape: (4, 4)
EXPERIMENT: 1544